In [1]:
import sys
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 200)

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "src" / "modeling.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find src/modeling.py from " + str(path))

ROOT = find_project_root()
sys.path.insert(0, str(ROOT / "src"))
import importlib
import modeling as m
importlib.reload(m)

undiagnosed = m.load("2017-2020")
all_adults = pd.read_csv(ROOT / "data" / "processed" / "cohort_all_2017-2020.csv")

print(f"Undiagnosed cohort: n = {len(undiagnosed)}, "
      f"positive = {undiagnosed['outcome_140_90'].mean():.1%}")
print(f"All adults:         n = {len(all_adults)}, "
      f"positive (diagnosed or BP >= 140/90) = {all_adults['outcome_any_140_90'].mean():.1%}")

candidates = {
    "Age only": ["age"],
    "Age + sex": ["age", "female"],
    "Model A": m.MODEL_A,
    "Model B": m.MODEL_B,
}
rows = {}
for name, predictors in candidates.items():
    a = m.summarise(m.cross_validate(m.make_logistic, predictors, undiagnosed,
                                     outcome="outcome_140_90"))
    b = m.summarise(m.cross_validate(m.make_logistic, predictors, all_adults,
                                     outcome="outcome_any_140_90"))
    rows[name] = {"AUC undiagnosed": a.loc["roc_auc", "mean"],
                  "AUC all adults": b.loc["roc_auc", "mean"]}

table = pd.DataFrame(rows).T
table["gain over age (undiagnosed)"] = table["AUC undiagnosed"] - table.loc["Age only", "AUC undiagnosed"]
table["gain over age (all adults)"] = table["AUC all adults"] - table.loc["Age only", "AUC all adults"]
print("\nROC-AUC, cross-validated (2017-2020)")
print(table.round(3))

# How different are diagnosed people? (descriptive)
print("\nMedian / % by diagnosis status (all adults)")
desc = all_adults.groupby("diagnosed").agg(
    n=("SEQN", "size"),
    age=("age", "median"),
    waist=("waist", "median"),
    pct_diabetes=("diabetes", lambda s: s.mean() * 100),
    hba1c=("hba1c", "median"),
)
desc.index = desc.index.map({0: "never told", 1: "diagnosed"})
print(desc.round(1))

Undiagnosed cohort: n = 4597, positive = 13.5%
All adults:         n = 7551, positive (diagnosed or BP >= 140/90) = 47.3%

ROC-AUC, cross-validated (2017-2020)
           AUC undiagnosed  AUC all adults  gain over age (undiagnosed)  gain over age (all adults)
Age only             0.733           0.770                        0.000                       0.000
Age + sex            0.736           0.770                        0.002                      -0.000
Model A              0.744           0.801                        0.011                       0.031
Model B              0.748           0.804                        0.014                       0.033

Median / % by diagnosis status (all adults)
               n   age  waist  pct_diabetes  hba1c
diagnosed                                         
never told  4597  43.0   95.9           7.7    5.5
diagnosed   2954  62.0  105.5          27.7    5.8
